# How degenerate is the flux distribution?

Standard FBA has alternative optima: many flux distributions give the same growth rate, and the
solver returns whichever vertex the simplex happens to land on. In a steady-state study that is a
footnote. In dFBA it is not, because the **exchange fluxes are what couple the cell to BioFVM** —
if the secretion profile is not unique, neither is the extracellular environment it produces, nor
the spatial pattern that follows from it.

This notebook measures that freedom instead of assuming it. For every distinct set of exchange
bounds the simulation actually generated, it rebuilds the cell's LP, pins the objective at its
optimum, and asks how far each exchange flux could move without changing growth. The width of
that interval is the degeneracy.

### Geometry

`cancer_core_metabolism` is **not a spheroid**. It is a thin slab of tumour tissue, 360 x 120 x 30
um at dx = 15, with Dirichlet conditions on the `xmin` face only (glucose and O2 at 0.5 mM, amino
acids at 1.0 mM) and `xmax` as a sink. Transport is effectively one-dimensional, so every spatial
result below is read against **x = depth into the tissue from the vessel**.

### Prerequisites

- a completed run whose configuration records the exchange lower bounds
  (`PhysiCell_settings_fva_sampling.xml`, which saves `full_data` every 10 min);
- `cobra`, `pandas`, `numpy`, `scipy`, `matplotlib`. On a machine with several Python
  installations, use the interpreter that can `import cobra`.


In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

sys.path.insert(0, os.path.dirname(os.path.abspath("fva_degeneracy.py")) or ".")
import fva_degeneracy as fd
from composite_figure_style import apply_cell_press_style

apply_cell_press_style()          # the project's journal typography

# An infeasible LP is an expected outcome here: it is a starving cell, not an error.
warnings.filterwarnings("ignore", message="Solver status is 'infeasible'")

# Point these at the run to analyse. The LP setup is read from the simulation's
# own configuration rather than written out here, so the replay cannot drift
# from the constraints the addon actually applied.
OUTPUT_DIR = "../../../../experiments/cancer_core_metabolism_fva_4320min/output"
CONFIG     = "../config/PhysiCell_settings_fva_sampling.xml"
SBML       = "../config/MCF7_Zielinski_2017_core_trimmed.sbml"

# For the acetate switch instead:
# OUTPUT_DIR = "../../../../experiments/ecoli_acetic_switch_fva_600min/output"
# CONFIG     = "../../ecoli_acetic_switch/config/PhysiCell_settings_fva_sampling.xml"
# SBML       = "../../ecoli_acetic_switch/config/Ecoli_core.xml"

EVERY      = 1        # keep one snapshot in EVERY; raise to subsample a long run
DECIMALS   = 10       # rounding applied before deduplicating bound sets

setup = fd.ModelSetup.from_config(CONFIG)

## 1. Load the run and deduplicate the bound sets

The Michaelis-Menten bound is a function of the local substrate concentration alone, not of the
cell, so every cell sharing a voxel at a given timepoint has an identical bound vector. The
exception is an exchange whose uptake was clamped to the amount actually present in the voxel,
which does depend on the cell's dry weight. Expect large compression, and report it.

In [ ]:
cells_df = fd.read_run(OUTPUT_DIR, every=EVERY)
uniq, inverse = fd.unique_bound_sets(cells_df, setup.exchanges, decimals=DECIMALS)

print(f"\ncells per snapshot: {cells_df.groupby('time').size().describe()[['min','max']].to_dict()}")
print(f"depth range sampled: x = {cells_df.x.min():.1f} to {cells_df.x.max():.1f} um")

## 2. Replay each bound set

The LP is rebuilt exactly as the addon constrains it: the recorded exchange lower bounds, the
objective reaction capped at `max_growth_rate`, and the death model's trigger flux given its
threshold as a lower bound. Omitting any of the three would answer a different question.

This is the expensive cell — roughly 46 LPs per unique bound set.

In [ ]:
model = fd.load_model(SBML)
fd.apply_simulation_constraints(model, setup)

fva = fd.fva_for_bound_sets(model, setup, uniq, fraction_of_optimum=1.0)

## 3. Self-check

Every flux the simulation recorded must lie inside its own FVA interval. If it does not, the
replay and the simulation disagree about the problem — a missing constraint, a wrong bound, or an
identifier mismatch — and nothing below should be trusted. This is the check that validates the
whole pipeline.

In [ ]:
check = fd.check_recorded_flux_within_range(cells_df, inverse, fva, setup)
display(check)

summary = fd.summarise(cells_df, inverse, fva, setup)
df = fd.attach_ranges(cells_df, inverse, fva, setup)
print(f"\nreplay status: {df.replay_status.value_counts().to_dict()}")
display(summary)

## 4. Which exchanges are degenerate?

Magnitude across categories, so a sorted horizontal bar. One series, so no legend — the axis
names it. Ranges span orders of magnitude, hence the log axis; the open marker is the 95th
percentile, showing how much worse the tail gets than the median.

In [ ]:
INK, MUTED, GRID = "#222222", "#6b6b6b", "#dddddd"
BAR  = "#0072B2"          # single hue: this figure encodes magnitude, not identity
FLOOR = 1e-9              # below this a range is solver noise, not degeneracy

s = summary[summary.median_range.notna()].copy()
s["plot_med"] = s.median_range.clip(lower=FLOOR)
s["plot_p95"] = s.p95_range.clip(lower=FLOOR)
s = s.sort_values("plot_med")

fig, ax = plt.subplots(figsize=(5.4, 0.26 * len(s) + 1.1), dpi=200)
y = np.arange(len(s))
at_floor = s.median_range.to_numpy() <= FLOOR
# A bar clipped to the floor means "no measurable degeneracy", not a small value.
# Colour it as absent so it cannot be misread as data.
ax.barh(y, s.plot_med, height=0.55, linewidth=0, zorder=3,
        color=np.where(at_floor, "#d9d9d9", BAR))
ax.scatter(s.plot_p95, y, s=16, facecolors="none", edgecolors=BAR,
           linewidths=1.0, zorder=4, label="95th percentile")

ax.set_yticks(y)
ax.set_yticklabels([r.replace("R_EX_", "").replace("_e", "") for r in s.reaction], fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("FVA interval width at fixed growth rate  (mmol gDW$^{-1}$ h$^{-1}$)", fontsize=8)
ax.axvline(FLOOR, color=MUTED, lw=0.7, ls=":", zorder=2)
ax.text(FLOOR, len(s) - 0.3, " solver noise floor", fontsize=6, color=MUTED, va="top")
ax.xaxis.grid(True, color=GRID, lw=0.5, zorder=0); ax.set_axisbelow(True)
for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
ax.tick_params(colors=INK, labelsize=7, length=2)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=BAR, label="median width"),
    Patch(facecolor="#d9d9d9", label="at noise floor"),
    plt.Line2D([], [], marker="o", ls="none", mfc="none", mec=BAR, ms=4,
               label="95th percentile")],
    frameon=False, fontsize=6.5, loc="lower right")
ax.set_title("Degeneracy of each exchange flux", fontsize=9, color=INK, loc="left")
fig.tight_layout(); plt.show()

## 5. Degeneracy against tissue depth

All 23 exchanges at once, so colour has to carry magnitude rather than identity: a single-hue
sequential ramp, light to dark. Rows are ordered by overall degeneracy, columns are depth bins
from the vessel. This is the figure that says *where* in the tissue the flux distribution is
underdetermined.

In [ ]:
NBINS = 12
ok = df[df.replay_status == "optimal"].copy()
edges = np.linspace(df.x.min(), df.x.max(), NBINS + 1)
ok["depth_bin"] = pd.cut(ok.x, bins=edges, include_lowest=True)
centers = 0.5 * (edges[:-1] + edges[1:])

order = [r for r in summary.sort_values("median_range", ascending=False).reaction
         if f"{r}_range" in ok.columns]
M = np.vstack([ok.groupby("depth_bin", observed=False)[f"{r}_range"].median()
               .reindex(ok.depth_bin.cat.categories).to_numpy() for r in order])

seq = LinearSegmentedColormap.from_list("seq", ["#f2f7fb", "#0072B2", "#00334f"])
seq.set_bad("#eeeeee")      # empty bin: no cells reached this depth, not zero width
M = np.ma.masked_invalid(M)
fig, ax = plt.subplots(figsize=(6.2, 0.24 * len(order) + 1.3), dpi=200)
im = ax.imshow(np.log10(np.clip(M, FLOOR, None)), aspect="auto", cmap=seq,
               origin="upper", extent=[edges[0], edges[-1], len(order) - 0.5, -0.5])
ax.set_yticks(range(len(order)))
ax.set_yticklabels([r.replace("R_EX_", "").replace("_e", "") for r in order], fontsize=7)
ax.set_xlabel("depth into tissue from the vessel, x  ($\\mu$m)", fontsize=8)
cb = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.035)
cb.set_label("log$_{10}$ median FVA width", fontsize=7); cb.ax.tick_params(labelsize=6)
ax.tick_params(colors=INK, labelsize=7, length=2)
ax.set_title("Where the flux distribution is underdetermined", fontsize=9, color=INK, loc="left")
ax.text(1.005, -0.09, "grey = no cells at that depth", transform=ax.transAxes,
        fontsize=6, color=MUTED, ha="right")
fig.tight_layout(); plt.show()

print("cells per depth bin:")
print(ok.groupby("depth_bin", observed=False).size().to_string())

## 6. The most degenerate exchanges over time

Six series at most, in a fixed order, from a palette validated for colour-vision deficiency
(worst adjacent pair dE 11.4 under protanopia, 18.4 for normal vision). Both a legend and direct
labels, so identity is never carried by colour alone.

In [ ]:
# fixed order, never cycled; validated with the dataviz palette checker
PALETTE = ["#0072B2", "#D55E00", "#56B4E9", "#009E73", "#E69F00", "#CC79A7"]
TOPN = 6
top = [r for r in summary.sort_values("median_range", ascending=False).reaction
       if f"{r}_range" in df.columns][:TOPN]

fig, ax = plt.subplots(figsize=(5.8, 3.2), dpi=200)
_placed = []
for colour, rid in zip(PALETTE, top):
    ts = ok.groupby("time")[f"{rid}_range"].median()
    label = rid.replace("R_EX_", "").replace("_e", "")
    ax.plot(ts.index / 60.0, ts.to_numpy(), color=colour, lw=2.0, label=label,
            solid_capstyle="round")
    if len(ts) and np.isfinite(ts.to_numpy()[-1]) and ts.to_numpy()[-1] > 0:
        yend = ts.to_numpy()[-1]
        # series can coincide exactly; offset the label so both stay readable
        while any(abs(np.log10(yend) - np.log10(p)) < 0.045 for p in _placed):
            yend *= 1.11
        _placed.append(yend)
        ax.annotate(f" {label}", (ts.index[-1] / 60.0, yend),
                    fontsize=6.5, color=colour, va="center", annotation_clip=False)

ax.set_xlabel("time (h)", fontsize=8)
ax.set_ylabel("median FVA width  (mmol gDW$^{-1}$ h$^{-1}$)", fontsize=8)
ax.set_yscale("log")
ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.tick_params(colors=INK, labelsize=7, length=2)
ax.legend(frameon=False, fontsize=7, ncol=3, loc="upper center",
          bbox_to_anchor=(0.5, -0.22))      # below the axes, never over the data
ax.set_title("Degeneracy as the nutrient gradient develops", fontsize=9, color=INK, loc="left")
fig.subplots_adjust(right=0.84, bottom=0.30); plt.show()

## 7. Save

`<prefix>_cells.csv` carries every cell-timepoint with its FVA interval attached, which is the
table a reviewer can be pointed at. `<prefix>_summary.csv` is the per-exchange ranking.

In [ ]:
df.to_csv("fva_degeneracy_cells.csv", index=False)
summary.to_csv("fva_degeneracy_summary.csv", index=False)
print("wrote fva_degeneracy_cells.csv and fva_degeneracy_summary.csv")

## How to read this

A width of zero means the flux was pinned: either it sat on its uptake bound, or the
stoichiometry left it no freedom. A width comparable to the flux itself means the simulation
picked one value out of a range of equally optimal ones, and a different solver, a different
pivot order, or a different PhysiCell build could legitimately have picked another.

Two caveats worth stating in any text built on this:

- The intervals are **marginal**, one reaction at a time. A large interval does not mean every
  value in it is reachable while the other fluxes stay put; the optimal face is a polytope, not a
  box. FVA bounds the projection onto each axis, which is the right statistic for asking "could
  this secretion have been different", and the wrong one for asking "could all of them have been
  different at once".
- `fraction_of_optimum = 1.0` asks for exactly the optimal face. Relaxing it slightly (0.99)
  widens every interval and is the more honest setting if the growth rate itself is only
  approximate — worth reporting both if the conclusion is sensitive to it.

Figures here are light-mode only, deliberately: they are built for print with the project's Cell
Press typography.
